In [ ]:
# ============================================================
# RESERVEIQ - COMPLETE PREPROCESSING
# SPLIT FIRST -> PREPROCESS AFTER SPLIT
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "is_canceled"

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# ============================================================
# 1. LOAD ORIGINAL RAW DATASET
# ============================================================

FILE_PATH = "hotel_booking.csv"

df = pd.read_csv(FILE_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Shape: (119390, 36)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,name,email,phone-number,credit_card
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,Ernest Barnes,Ernest.Barnes31@outlook.com,669-792-1661,************4322
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,Andrea Baker,Andrea_Baker94@aol.com,858-637-6955,************9157
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,Rebecca Parker,Rebecca_Parker@comcast.net,652-885-2745,************3734
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,Laura Murray,Laura_M@gmail.com,364-656-8427,************5677
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,Linda Hines,LHines@verizon.com,713-226-5883,************5498


In [ ]:
# ============================================================
# 2. CHECK TARGET
# ============================================================

print("Target values:")
print(df[TARGET].value_counts())

print("\nTarget percentage:")
print(
    df[TARGET]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nUnique target values:")
print(df[TARGET].unique())

Target values:
is_canceled
0    75166
1    44224
Name: count, dtype: int64

Target percentage:
is_canceled
0    62.96
1    37.04
Name: proportion, dtype: float64

Unique target values:
[0 1]


In [ ]:
# ============================================================
# 3. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(int).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (119390, 35)
y shape: (119390,)

Target distribution:
is_canceled
0    75166
1    44224
Name: count, dtype: int64


In [ ]:
# ============================================================
# 4. STRATIFIED TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# Reset indices to keep everything aligned
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("TRAIN / TEST SPLIT COMPLETE")

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining target %:")
print(
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nTesting target %:")
print(
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

TRAIN / TEST SPLIT COMPLETE

X_train: (95512, 35)
X_test : (23878, 35)
y_train: (95512,)
y_test : (23878,)

Training target %:
is_canceled
0    62.96
1    37.04
Name: proportion, dtype: float64

Testing target %:
is_canceled
0    62.96
1    37.04
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# MEMBER 1
# MISSING VALUES AND DATA QUALITY
# ============================================================

print("=" * 70)
print("MEMBER 1 - MISSING VALUES AND DATA QUALITY")
print("=" * 70)

print("\nTRAIN missing values:")
print(
    X_train.isnull().sum()[
        X_train.isnull().sum() > 0
    ].sort_values(ascending=False)
)

print("\nTEST missing values:")
print(
    X_test.isnull().sum()[
        X_test.isnull().sum() > 0
    ].sort_values(ascending=False)
)

MEMBER 1 - MISSING VALUES AND DATA QUALITY

TRAIN missing values:
company     90111
agent       13034
country       391
children        4
dtype: int64

TEST missing values:
company    22482
agent       3306
country       97
dtype: int64


In [ ]:
# ============================================================
# 5. HANDLE COMPANY
# ============================================================

for data in [X_train, X_test]:

    data["company"] = data["company"].apply(
        lambda x:
        "No Company"
        if pd.isna(x)
        else str(int(x))
    )

print("Company missing:")
print("Train:", X_train["company"].isnull().sum())
print("Test :", X_test["company"].isnull().sum())

Company missing:
Train: 0
Test : 0


In [ ]:
# ============================================================
# 6. HANDLE AGENT
# ============================================================

for data in [X_train, X_test]:

    data["agent"] = data["agent"].apply(
        lambda x:
        "No Agent"
        if pd.isna(x)
        else str(int(x))
    )

print("Agent missing:")
print("Train:", X_train["agent"].isnull().sum())
print("Test :", X_test["agent"].isnull().sum())

Agent missing:
Train: 0
Test : 0


In [ ]:
# ============================================================
# 7. HANDLE COUNTRY
# ============================================================

X_train["country"] = X_train["country"].fillna("Unknown")
X_test["country"] = X_test["country"].fillna("Unknown")

print("Country missing:")
print("Train:", X_train["country"].isnull().sum())
print("Test :", X_test["country"].isnull().sum())

Country missing:
Train: 0
Test : 0


In [ ]:
# ============================================================
# 8. HANDLE CHILDREN - TRAIN-ONLY MEDIAN
# ============================================================

children_median = X_train["children"].median()

print("Training children median:", children_median)

X_train["children"] = X_train["children"].fillna(
    children_median
)

X_test["children"] = X_test["children"].fillna(
    children_median
)

print("\nChildren missing after imputation:")
print("Train:", X_train["children"].isnull().sum())
print("Test :", X_test["children"].isnull().sum())

Training children median: 0.0

Children missing after imputation:
Train: 0
Test : 0


In [ ]:
# ============================================================
# 9. DATA QUALITY CHECKS
# ============================================================

def quality_report(data, name):

    total_guests = (
        data["adults"]
        + data["children"]
        + data["babies"]
    )

    total_nights = (
        data["stays_in_weekend_nights"]
        + data["stays_in_week_nights"]
    )

    report = {
        "Rows": len(data),

        "Zero total guests":
            int((total_guests == 0).sum()),

        "Negative adults":
            int((data["adults"] < 0).sum()),

        "Negative children":
            int((data["children"] < 0).sum()),

        "Negative babies":
            int((data["babies"] < 0).sum()),

        "Zero total nights":
            int((total_nights == 0).sum()),

        "Negative ADR":
            int((data["adr"] < 0).sum()),

        "Invalid arrival day":
            int(
                (
                    (data["arrival_date_day_of_month"] < 1)
                    |
                    (data["arrival_date_day_of_month"] > 31)
                ).sum()
            ),

        "Invalid arrival week":
            int(
                (
                    (data["arrival_date_week_number"] < 1)
                    |
                    (data["arrival_date_week_number"] > 53)
                ).sum()
            )
    }

    print("\n", name)
    print(pd.Series(report))


quality_report(X_train, "TRAIN DATA QUALITY")
quality_report(X_test, "TEST DATA QUALITY")


 TRAIN DATA QUALITY
Rows                    95512
Zero total guests         145
Negative adults             0
Negative children           0
Negative babies             0
Zero total nights         557
Negative ADR                1
Invalid arrival day         0
Invalid arrival week        0
dtype: int64

 TEST DATA QUALITY
Rows                    23878
Zero total guests          35
Negative adults             0
Negative children           0
Negative babies             0
Zero total nights         158
Negative ADR                0
Invalid arrival day         0
Invalid arrival week        0
dtype: int64


In [ ]:
# ============================================================
# 10. NEGATIVE ADR TREATMENT
# ============================================================

print("Negative ADR before treatment:")

print(
    "Train:",
    (X_train["adr"] < 0).sum()
)

print(
    "Test:",
    (X_test["adr"] < 0).sum()
)

# Negative ADR -> NaN
X_train.loc[X_train["adr"] < 0, "adr"] = np.nan
X_test.loc[X_test["adr"] < 0, "adr"] = np.nan

Negative ADR before treatment:
Train: 1
Test: 0


In [ ]:
adr_median = X_train["adr"].median()

print("Training ADR median:", adr_median)

X_train["adr"] = X_train["adr"].fillna(
    adr_median
)

X_test["adr"] = X_test["adr"].fillna(
    adr_median
)

Training ADR median: 94.5


In [ ]:
# ============================================================
# MEMBER 2
# FEATURE REMOVAL / PRIVACY / TARGET LEAKAGE
# ============================================================

print("=" * 70)
print("MEMBER 2 - FEATURE REMOVAL")
print("=" * 70)

drop_columns = [
    "name",
    "email",
    "phone-number",
    "credit_card",
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type"
]

drop_columns = [
    col
    for col in drop_columns
    if col in X_train.columns
]

X_train = X_train.drop(
    columns=drop_columns
)

X_test = X_test.drop(
    columns=drop_columns
)

print("Removed columns:")
print(drop_columns)

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)

MEMBER 2 - FEATURE REMOVAL
Removed columns:
['name', 'email', 'phone-number', 'credit_card', 'reservation_status', 'reservation_status_date', 'assigned_room_type']

Train shape: (95512, 28)
Test shape : (23878, 28)


In [ ]:
# ============================================================
# MEMBER 3
# FEATURE ENGINEERING
# ============================================================

print("=" * 70)
print("MEMBER 3 - FEATURE ENGINEERING")
print("=" * 70)

for data in [X_train, X_test]:

    # Total guests
    data["total_guests"] = (
        data["adults"]
        + data["children"]
        + data["babies"]
    )

    # Total nights
    data["total_nights"] = (
        data["stays_in_weekend_nights"]
        + data["stays_in_week_nights"]
    )

print("Engineered features:")
print("1. total_guests")
print("2. total_nights")

MEMBER 3 - FEATURE ENGINEERING
Engineered features:
1. total_guests
2. total_nights


In [ ]:
important_numeric = [
    "lead_time",
    "is_repeated_guest",
    "total_of_special_requests",
    "total_guests",
    "total_nights"
]

print("Important numeric features:")

for col in important_numeric:
    print(
        col,
        "→",
        "PRESENT"
        if col in X_train.columns
        else "MISSING"
    )

Important numeric features:
lead_time → PRESENT
is_repeated_guest → PRESENT
total_of_special_requests → PRESENT
total_guests → PRESENT
total_nights → PRESENT


In [ ]:
# ============================================================
# 11. CHECK MISSING VALUES
# ============================================================

print("TRAIN remaining missing:")
print(
    X_train.isnull().sum()[
        X_train.isnull().sum() > 0
    ]
)

print("\nTEST remaining missing:")
print(
    X_test.isnull().sum()[
        X_test.isnull().sum() > 0
    ]
)

TRAIN remaining missing:
Series([], dtype: int64)

TEST remaining missing:
Series([], dtype: int64)


In [ ]:
# ============================================================
# MEMBER 4
# HIGH-CARDINALITY FREQUENCY ENCODING
# ============================================================

print("=" * 70)
print("MEMBER 4 - CATEGORICAL ENCODING")
print("=" * 70)

high_cardinality_cols = [
    "country",
    "agent",
    "company"
]

frequency_maps = {}

for col in high_cardinality_cols:

    if col in X_train.columns:

        # Learn frequencies from TRAINING DATA ONLY
        freq_map = (
            X_train[col]
            .value_counts(normalize=True)
        )

        frequency_maps[col] = freq_map

        # Apply to training data
        X_train[col + "_freq"] = (
            X_train[col]
            .map(freq_map)
            .fillna(0)
        )

        # Apply SAME map to test data
        X_test[col + "_freq"] = (
            X_test[col]
            .map(freq_map)
            .fillna(0)
        )

        # Remove original high-cardinality column
        X_train.drop(
            columns=[col],
            inplace=True
        )

        X_test.drop(
            columns=[col],
            inplace=True
        )

print("Frequency encoding complete.")

print("\nCreated:")
for col in high_cardinality_cols:
    print(col + "_freq")

MEMBER 4 - CATEGORICAL ENCODING
Frequency encoding complete.

Created:
country_freq
agent_freq
company_freq


In [ ]:
# ============================================================
# 12. IDENTIFY CATEGORICAL / NUMERIC FEATURES
# ============================================================

categorical_cols = (
    X_train
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

numeric_cols = (
    X_train
    .select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)

print("Categorical columns:")
print(categorical_cols)

print("\nNumeric columns:")
print(numeric_cols)

print("\nNumber categorical:",
      len(categorical_cols))

print("Number numeric:",
      len(numeric_cols))

Categorical columns:
['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type']

Numeric columns:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'total_nights', 'country_freq', 'agent_freq', 'company_freq']

Number categorical: 8
Number numeric: 22


In [ ]:
# ============================================================
# 13. NUMERIC IMPUTATION - TRAIN ONLY
# ============================================================

numeric_imputer = SimpleImputer(
    strategy="median"
)

X_train[numeric_cols] = (
    numeric_imputer.fit_transform(
        X_train[numeric_cols]
    )
)

X_test[numeric_cols] = (
    numeric_imputer.transform(
        X_test[numeric_cols]
    )
)

print("Numeric imputation complete.")

Numeric imputation complete.


In [ ]:
# ============================================================
# 14. ONE-HOT ENCODING
# ============================================================

onehot = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Learn categories from TRAIN only
train_encoded = onehot.fit_transform(
    X_train[categorical_cols]
)

# Use same encoder on TEST
test_encoded = onehot.transform(
    X_test[categorical_cols]
)

encoded_feature_names = (
    onehot.get_feature_names_out(
        categorical_cols
    )
)

print(
    "Number of encoded categorical features:",
    len(encoded_feature_names)
)

Number of encoded categorical features: 49


In [ ]:
# ============================================================
# 15. CREATE ENCODED DATAFRAMES
# ============================================================

train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

print("Train encoded shape:",
      train_encoded_df.shape)

print("Test encoded shape:",
      test_encoded_df.shape)

Train encoded shape: (95512, 49)
Test encoded shape: (23878, 49)


In [ ]:
# ============================================================
# 16. REMOVE ORIGINAL CATEGORICAL COLUMNS
# ============================================================

X_train_numeric = X_train.drop(
    columns=categorical_cols
).copy()

X_test_numeric = X_test.drop(
    columns=categorical_cols
).copy()

In [ ]:
# ============================================================
# 17. COMBINE NUMERIC + ENCODED FEATURES
# ============================================================

X_train_processed = pd.concat(
    [
        X_train_numeric.reset_index(drop=True),
        train_encoded_df.reset_index(drop=True)
    ],
    axis=1
)

X_test_processed = pd.concat(
    [
        X_test_numeric.reset_index(drop=True),
        test_encoded_df.reset_index(drop=True)
    ],
    axis=1
)

print("Processed train shape:",
      X_train_processed.shape)

print("Processed test shape:",
      X_test_processed.shape)

Processed train shape: (95512, 71)
Processed test shape: (23878, 71)


In [ ]:
# ============================================================
# 18. PREPROCESSING VERIFICATION
# ============================================================

print("=" * 70)
print("PREPROCESSING VERIFICATION")
print("=" * 70)

print(
    "Same columns:",
    list(X_train_processed.columns)
    ==
    list(X_test_processed.columns)
)

print(
    "Train missing values:",
    X_train_processed.isnull().sum().sum()
)

print(
    "Test missing values:",
    X_test_processed.isnull().sum().sum()
)

print(
    "Non-numeric train columns:",
    X_train_processed
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print(
    "Non-numeric test columns:",
    X_test_processed
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print("\nTrain:", X_train_processed.shape)
print("Test :", X_test_processed.shape)

print("\ny_train:", y_train.shape)
print("y_test :", y_test.shape)

PREPROCESSING VERIFICATION
Same columns: True
Train missing values: 0
Test missing values: 0
Non-numeric train columns: []
Non-numeric test columns: []

Train: (95512, 71)
Test : (23878, 71)

y_train: (95512,)
y_test : (23878,)


In [ ]:
# ============================================================
# 19. TEMPORARY RANDOM FOREST FOR FEATURE SELECTION
# ============================================================

rf_feature_selector = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# TRAIN DATA ONLY
rf_feature_selector.fit(
    X_train_processed,
    y_train
)

print(
    "Temporary feature-selection Random Forest trained."
)

Temporary feature-selection Random Forest trained.


In [ ]:
# ============================================================
# 20. FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    "Feature":
        X_train_processed.columns,

    "Importance":
        rf_feature_selector.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    feature_importance.head(30)
)

,Feature,Importance
0,country_freq,0.106845
1,lead_time,0.106115
2,deposit_type_No Deposit,0.074620
3,deposit_type_Non Refund,0.072481
4,adr,0.063689
5,total_of_special_requests,0.055886
6,agent_freq,0.049449
7,arrival_date_day_of_month,0.048661
8,arrival_date_week_number,0.042693
9,previous_cancellations,0.031944


In [ ]:
# ============================================================
# SAVE FULL PREPROCESSED DATA - BEFORE FEATURE SELECTION
# ============================================================

# Keep ALL processed features
X_train_all_features = X_train_processed.copy()
X_test_all_features = X_test_processed.copy()

# Save feature datasets
X_train_all_features.to_csv(
    "X_train_all_features.csv",
    index=False
)

X_test_all_features.to_csv(
    "X_test_all_features.csv",
    index=False
)

# Save targets
y_train.to_csv(
    "y_train_all_features.csv",
    index=False
)

y_test.to_csv(
    "y_test_all_features.csv",
    index=False
)

print("Full preprocessed datasets saved successfully.")
print("NO feature selection has been applied.")

print("\nX_train:", X_train_all_features.shape)
print("X_test :", X_test_all_features.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nNumber of features:",
      X_train_all_features.shape[1])

Full preprocessed datasets saved successfully.
NO feature selection has been applied.

X_train: (95512, 71)
X_test : (23878, 71)
y_train: (95512,)
y_test : (23878,)

Number of features: 71


In [ ]:
# ============================================================
# 21. SELECT TOP 20 FEATURES
# ============================================================

TOP_N = 20

top_features = (
    feature_importance
    .head(TOP_N)["Feature"]
    .tolist()
)

print("TOP 20 FEATURES")

for i, feature in enumerate(
    top_features,
    start=1
):
    print(i, feature)

TOP 20 FEATURES
1 country_freq
2 lead_time
3 deposit_type_No Deposit
4 deposit_type_Non Refund
5 adr
6 total_of_special_requests
7 agent_freq
8 arrival_date_day_of_month
9 arrival_date_week_number
10 previous_cancellations
11 total_nights
12 stays_in_week_nights
13 arrival_date_year
14 required_car_parking_spaces
15 booking_changes
16 stays_in_weekend_nights
17 customer_type_Transient-Party
18 market_segment_Groups
19 market_segment_Online TA
20 customer_type_Transient


In [ ]:
# ============================================================
# 22. FINAL FEATURE-SELECTED DATASETS
# ============================================================

X_train_final = (
    X_train_processed[
        top_features
    ].copy()
)

X_test_final = (
    X_test_processed[
        top_features
    ].copy()
)

print("FINAL DATA")

print(
    "X_train_final:",
    X_train_final.shape
)

print(
    "X_test_final :",
    X_test_final.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "y_test :",
    y_test.shape
)

NameError: name 'top_features' is not defined

In [ ]:
# ============================================================
# 23. FINAL SAFETY CHECK
# ============================================================

print("=" * 70)
print("FINAL PREPROCESSING CHECK")
print("=" * 70)

print(
    "Train/Test columns match:",
    list(X_train_final.columns)
    ==
    list(X_test_final.columns)
)

print(
    "Train missing:",
    X_train_final.isnull().sum().sum()
)

print(
    "Test missing:",
    X_test_final.isnull().sum().sum()
)

print(
    "Train samples match:",
    len(X_train_final)
    ==
    len(y_train)
)

print(
    "Test samples match:",
    len(X_test_final)
    ==
    len(y_test)
)

print("\nFinal target distribution - TRAIN")

print(
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nFinal target distribution - TEST")

print(
    y_test
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

FINAL PREPROCESSING CHECK


NameError: name 'X_train_final' is not defined

In [ ]:
# ============================================================
# 24. DATA READY FOR RANDOM FOREST
# ============================================================

X_train = X_train_final.copy()
X_test = X_test_final.copy()

print("READY FOR RANDOM FOREST")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

NameError: name 'X_train_final' is not defined

In [ ]:
# ============================================================
# 25. SAVE FINAL PREPROCESSED DATA
# ============================================================

X_train.to_csv(
    "X_train_preprocessed.csv",
    index=False
)

X_test.to_csv(
    "X_test_preprocessed.csv",
    index=False
)

y_train.to_csv(
    "y_train.csv",
    index=False
)

y_test.to_csv(
    "y_test.csv",
    index=False
)

feature_importance.to_csv(
    "preprocessing_feature_importance.csv",
    index=False
)

print("Files saved successfully.")

Files saved successfully.
